### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
# from langchain_ollama import ChatOllama
# model1=ChatOllama(model="gemma:2b")
# print(model1.invoke("What is the capital of France?"))

from langchain_groq import ChatGroq
model=ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
model

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x000002AB02D87B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AB02F38980>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to Hindi"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [4]:
result

AIMessage(content='नमस्ते, आप कैसे हैं?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 261, 'total_tokens': 337, 'completion_time': 0.167259, 'completion_tokens_details': None, 'prompt_time': 0.009928, 'prompt_tokens_details': None, 'queue_time': 0.097546, 'total_time': 0.177187}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7c9d-d7c1-7323-a786-47081145b006-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 261, 'output_tokens': 76, 'total_tokens': 337})

In [5]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'नमस्ते, आप कैसे हैं?'

In [6]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'नमस्ते, आप कैसे हैं?'

In [7]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)


In [8]:
result=prompt.invoke({"language":"Hindi","text":"Hello"})

In [9]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into Hindi:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [10]:
chain=prompt|model|parser
chain.invoke({"language":"Hindi","text":"Hello"})

'नमस्ते'